# Stage 5 §V0 — off-policy rollout distillation into the value head (Colab T4)

**Question.** §H.7 diagnosed the value head's failure as OFF-POLICY calibration: perfectly calibrated on trajectory states (|err| 0.08 raw) but ~0.6 raw optimistic on untaken sibling children — exactly where MCTS reads it — so its sibling-ranking signal is ≈ 0. V0 tests the fix: **distill the greedy rollout into the head on those counterfactual children** (labels the search already computes for free under rollout self-play). If ranking is fixed, the head becomes an *amortized rollout* — the compute play of plan §12 (≈ 1+N/2× sims at matched wall).

**Pipeline** (all on the frozen §H.4 policy — no self-play, no C++ changes):
1. `build_offpolicy_value_dataset` — rollout-label ALL legal children of ~20k buffer states (train split `inst%5!=0`), ~200k pairs.
2. `train_value_head_offpolicy` — MSE-distill into the glimpse head (policy frozen; 16.6k trainable params).
3. Gate probes on HELD-OUT instances (`inst%5==0`):
   - **G1 (depth-1)**: `probe_action_ranking` vs the committed gt cache. **PASS = decision regret ≤ 0.08 raw AND intrinsic Spearman(v,g) ≥ 0.5.**
   - **G2 (depth-2, root_hop=1)**: same probe 1 random step off-trajectory (fresh gt generated here). **PASS = regret ≤ 2× the rollout anchor on the same probe AND Spearman(v,g) ≥ 0.4.**

**Anchors** (TSP-20, 50 nodes, sampled-E[z|s'] gt): greedy rollout regret **0.047 (all) / 0.056 (held-out)**, Spearman 0.89, top-1 0.74. Pre-distillation head: regret 0.199, Spearman(v,g) 0.081 (held-out). Local CPU smoke with only 219 pairs already moved held-out Spearman(v,g) 0.081 → 0.489.

**Wall estimate on T4**: dataset ~5 min + training ~10-15 min + probes ~10 min ≈ **30-40 min total**.

**Requires on Drive**: the §H.4 run dir `outputs/tsp_20/tsp20_k10_mix0p5_step50_100iter_20260522T055640_20260522T055644/` with `iter-99.pt`, `buffer.pt`, `args.json` (already there from the Phase B run).

## Section 1 — setup (Drive mount + repo + install)

In [ ]:
import sys, platform
print('python    =', sys.version.split()[0], platform.platform())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch     =', torch.__version__, '  cuda available =', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AM_AlphaGoZero'
REPO_DIR = os.path.join(WORKSPACE, 'repo')
OUTPUT_DIR = os.path.join(WORKSPACE, 'outputs')
RUN_DIR = os.path.join(OUTPUT_DIR, 'tsp_20', 'tsp20_k10_mix0p5_step50_100iter_20260522T055640_20260522T055644')

CKPT = os.path.join(RUN_DIR, 'iter-99.pt')
BUFFER = os.path.join(RUN_DIR, 'buffer.pt')
for f in (CKPT, BUFFER, os.path.join(RUN_DIR, 'args.json')):
    assert os.path.exists(f), f'missing on Drive: {f}'
print('RUN_DIR =', RUN_DIR)
print('OK — iter-99.pt / buffer.pt / args.json present')

In [ ]:
REPO_URL = 'https://github.com/LejunZhou/AM_ALPHAGOZERO.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned; pulling latest...')
    !git -C {REPO_DIR} pull --ff-only

!git -C {REPO_DIR} log --oneline -1

In [ ]:
%cd {REPO_DIR}
!pip install --quiet pybind11
!pip install --quiet --no-deps -e .
!pip install --quiet numpy scipy tqdm

import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
for name in [n for n in list(sys.modules) if n.startswith('am_baseline') or n.startswith('scripts')]:
    del sys.modules[name]

# V0 scripts present?
for s in ('build_offpolicy_value_dataset', 'train_value_head_offpolicy',
          'probe_action_ranking', 'rank_rollout_benchmark'):
    assert os.path.exists(os.path.join(REPO_DIR, 'src', 'scripts', s + '.py')), f'{s}.py missing — pull latest'

# Copy the committed gt caches next to the checkpoint (probes take --gt_cache paths there).
import shutil
GT_SRC = os.path.join(REPO_DIR, '_progress', 'eval_logs', 'rank_gt')
GT_EVAL_D1 = os.path.join(RUN_DIR, 'rank_gt_1b68dbd028.npz')   # depth-1, held-out split, sampled gt
for f in os.listdir(GT_SRC):
    dst = os.path.join(RUN_DIR, f)
    if not os.path.exists(dst):
        shutil.copy(os.path.join(GT_SRC, f), dst)
print('OK — scripts + gt caches in place')

## Section 2 — build the off-policy datasets (~5 min)

Train split (`inst%5!=0`): ~19 states/step × 19 steps × ALL legal children ≈ 200k rollout-labeled pairs. Eval split (`inst%5==0`): 2k states for held-out MSE monitoring during training (the real gate is Section 4).

In [ ]:
DS_TRAIN = os.path.join(RUN_DIR, 'offpolicy_value_ds_train.pt')
DS_EVAL = os.path.join(RUN_DIR, 'offpolicy_value_ds_eval.pt')

!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.build_offpolicy_value_dataset \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --num_states 20000 --children_per_state 0 \
    --holdout_k 5 --split train --seed 1234 --out {DS_TRAIN}

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.build_offpolicy_value_dataset \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --num_states 2000 --children_per_state 0 \
    --holdout_k 5 --split eval --seed 1235 --out {DS_EVAL}

## Section 3 — distill (~10-15 min)

Glimpse-head MLP only (16.6k params), warm-started from the §H.4 head, policy frozen. Watch `heldout_rmse`: it starts ≈ 1.0 (the §H.7.2 off-policy error) and should fall well below 0.3.

In [ ]:
VH_CKPT = os.path.join(RUN_DIR, 'iter-99_vh_offpolicy.pt')

!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.train_value_head_offpolicy \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --dataset {DS_TRAIN} --eval_dataset {DS_EVAL} \
    --epochs 20 --steps_per_epoch 400 --batch_size 512 \
    --lr 1e-3 --lr_decay 0.2 --lr_decay_step 10 --max_grad_norm 1.0 \
    --seed 1234 --out_ckpt {VH_CKPT}

## Section 4 — Gate G1: depth-1 ranking on held-out instances

Reuses the committed gt cache (identical slots ⇒ numbers directly comparable to §H.7.1). Probes the distilled head, then the ORIGINAL head side-by-side, then prints the rollout anchor.

In [ ]:
CSV_G1_NEW = os.path.join(RUN_DIR, 'rank_vh_offpolicy_eval.csv')
CSV_G1_OLD = os.path.join(RUN_DIR, 'rank_glimpse_orig_eval.csv')

!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.probe_action_ranking \
    --ckpt {VH_CKPT} --buffer {BUFFER} --which best \
    --num_nodes 50 --rollouts_per_child 16 --min_actions 4 \
    --holdout_k 5 --inst_split eval --seed 1234 \
    --gt_cache {GT_EVAL_D1} --out_csv {CSV_G1_NEW}

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.probe_action_ranking \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --num_nodes 50 --rollouts_per_child 16 --min_actions 4 \
    --holdout_k 5 --inst_split eval --seed 1234 \
    --gt_cache {GT_EVAL_D1} --out_csv {CSV_G1_OLD}

# Rollout anchor (cache arithmetic, zero model calls)
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.rank_rollout_benchmark \
    --gt_greedy {os.path.join(RUN_DIR, 'rank_gt_bdec4956cc.npz')} \
    --gt_sampled {GT_EVAL_D1}

## Section 5 — Gate G2: depth-2 (root_hop=1) generalization (~10 min)

Nodes pushed 1 seeded-random step off the buffer trajectory — states neither the buffer nor the depth-1 training children cover exactly. Generates fresh sampled + greedy gt (policy identical across checkpoints, so the caches are shared), computes the rollout anchor on the SAME probe, then ranks both heads.

In [ ]:
GT_H1_SAMPLED = os.path.join(RUN_DIR, 'rank_gt_hop1_eval_sampled.npz')
GT_H1_GREEDY = os.path.join(RUN_DIR, 'rank_gt_hop1_eval_greedy.npz')
CSV_G2_NEW = os.path.join(RUN_DIR, 'rank_vh_offpolicy_eval_hop1.csv')
CSV_G2_OLD = os.path.join(RUN_DIR, 'rank_glimpse_orig_eval_hop1.csv')

# Sampled gt (generated on the first probe run; ~8 min on T4) + distilled head ranking.
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.probe_action_ranking \
    --ckpt {VH_CKPT} --buffer {BUFFER} --which best \
    --num_nodes 50 --rollouts_per_child 16 --min_actions 4 --root_hop 1 \
    --holdout_k 5 --inst_split eval --seed 1234 --gt_mode sampled \
    --gt_cache {GT_H1_SAMPLED} --out_csv {CSV_G2_NEW}

In [ ]:
# Original head vs the SAME sampled gt (cache hit — fast).
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.probe_action_ranking \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --num_nodes 50 --rollouts_per_child 16 --min_actions 4 --root_hop 1 \
    --holdout_k 5 --inst_split eval --seed 1234 --gt_mode sampled \
    --gt_cache {GT_H1_SAMPLED} --out_csv {CSV_G2_OLD}

# Greedy gt for the rollout anchor (1 rollout/child; ~1 min; CSV is a byproduct).
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.probe_action_ranking \
    --ckpt {CKPT} --buffer {BUFFER} --which best \
    --num_nodes 50 --rollouts_per_child 16 --min_actions 4 --root_hop 1 \
    --holdout_k 5 --inst_split eval --seed 1234 --gt_mode greedy \
    --gt_cache {GT_H1_GREEDY} --out_csv {os.path.join(RUN_DIR, 'rank_tmp_hop1_greedygt.csv')}

# Rollout anchor at depth-2 — gate G2's reference point.
!cd {REPO_DIR} && PYTHONPATH=src python -m scripts.rank_rollout_benchmark \
    --gt_greedy {GT_H1_GREEDY} --gt_sampled {GT_H1_SAMPLED}

## Section 6 — verdict

In [ ]:
import csv
import numpy as np

def summary(path):
    with open(path, newline='') as f:
        rows = list(csv.DictReader(f))
    def m(k):
        v = [float(r[k]) for r in rows if r[k] not in ('', 'nan')]
        return float(np.mean([x for x in v if not np.isnan(x)]))
    return {'regret': m('regret_raw'), 'sp_ctg': m('spearman_ctg'),
            'sp_action': m('spearman_action'), 'top1': m('top1'), 'n': len(rows)}

g1_new, g1_old = summary(CSV_G1_NEW), summary(CSV_G1_OLD)
g2_new, g2_old = summary(CSV_G2_NEW), summary(CSV_G2_OLD)

# Rollout anchor at depth-2, recomputed inline for the gate arithmetic.
zg, zs = np.load(GT_H1_GREEDY), np.load(GT_H1_SAMPLED)
anchor2 = []
for nid in np.unique(zg['node_id']):
    mg, ms = zg['node_id'] == nid, zs['node_id'] == nid
    avg, avs = zg['edge'][mg] + zg['gmean'][mg], zs['edge'][ms] + zs['gmean'][ms]
    anchor2.append(float(avs[int(np.argmin(avg))] - avs.min()))
ROLLOUT_D1, ROLLOUT_D2 = 0.05635, float(np.mean(anchor2))

print(f"{'probe':<34}{'regret':>9}{'sp(v,g)':>9}{'sp(act)':>9}{'top1':>7}")
print(f"{'rollout anchor depth-1':<34}{ROLLOUT_D1:>9.4f}{'--':>9}{0.879:>9.3f}{0.74:>7.2f}")
print(f"{'head ORIG depth-1':<34}{g1_old['regret']:>9.4f}{g1_old['sp_ctg']:>9.3f}{g1_old['sp_action']:>9.3f}{g1_old['top1']:>7.2f}")
print(f"{'head DISTILLED depth-1':<34}{g1_new['regret']:>9.4f}{g1_new['sp_ctg']:>9.3f}{g1_new['sp_action']:>9.3f}{g1_new['top1']:>7.2f}")
print(f"{'rollout anchor depth-2 (hop1)':<34}{ROLLOUT_D2:>9.4f}")
print(f"{'head ORIG depth-2':<34}{g2_old['regret']:>9.4f}{g2_old['sp_ctg']:>9.3f}{g2_old['sp_action']:>9.3f}{g2_old['top1']:>7.2f}")
print(f"{'head DISTILLED depth-2':<34}{g2_new['regret']:>9.4f}{g2_new['sp_ctg']:>9.3f}{g2_new['sp_action']:>9.3f}{g2_new['top1']:>7.2f}")

G1 = g1_new['regret'] <= 0.08 and g1_new['sp_ctg'] >= 0.5
G2 = g2_new['regret'] <= 2 * ROLLOUT_D2 and g2_new['sp_ctg'] >= 0.4
print(f"\nGATE G1 (depth-1: regret<=0.08 & sp_ctg>=0.5):        {'PASS' if G1 else 'FAIL'}")
print(f"GATE G2 (depth-2: regret<=2x{ROLLOUT_D2:.3f} & sp_ctg>=0.4): {'PASS' if G2 else 'FAIL'}")
print('\nVERDICT:', 'V0 PASS — proceed to V1 (matched-wall vh-only search on TSP-20)' if (G1 and G2)
      else 'V0 PARTIAL/FAIL — paste the table back; next levers: more states, hop-augmented training data, trunk head')

## Next steps

Paste the Section 6 table + verdict back to the main thread. Artifacts persisted on Drive in `RUN_DIR`: `iter-99_vh_offpolicy.pt`, both datasets, 4 probe CSVs, 2 new hop-1 gt caches. I'll record the result in `_progress/stage5_offpolicy_value_progress.md` §V0 and, on PASS, scaffold V1 (matched-wall `leaf_eval=value_head` val on TSP-20: K≈400 vs rollout K=40 at equal wall).